[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/pypath/blob/main/notebooks/module4/04-visualization.ipynb)

# Data Visualization: Matplotlib and Seaborn

**Module 4 — Data Science & Visualization** | Estimated time: 30 minutes

## Learning Objectives

By the end of this notebook you will be able to:
- Create and customize figures, axes, and subplots with Matplotlib
- Build line, bar, scatter, histogram, and pie charts
- Apply styles, color palettes, labels, and legends
- Use Seaborn for statistical plots: histplot, boxplot, violinplot, heatmap, pairplot
- Use FacetGrid for multi-panel plots
- Save publication-quality figures to disk

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# Set global style defaults
plt.rcParams.update({'figure.dpi': 110, 'figure.facecolor': 'white'})
sns.set_theme(style='whitegrid', palette='tab10')

print(f'Matplotlib: {plt.matplotlib.__version__}')
print(f'Seaborn:    {sns.__version__}')

rng = np.random.default_rng(0)

## 1. Matplotlib Fundamentals: Figure and Axes

Matplotlib uses a **Figure** as the top-level container and **Axes** as individual plot panels. `plt.subplots()` returns both and is the recommended way to create multi-panel figures.

In [ ]:
x = np.linspace(0, 10, 300)

fig, axes = plt.subplots(2, 3, figsize=(14, 7))

# Line chart
axes[0, 0].plot(x, np.sin(x), color='steelblue', linewidth=2, label='sin(x)')
axes[0, 0].plot(x, np.cos(x), color='tomato',    linewidth=2, label='cos(x)', linestyle='--')
axes[0, 0].set_title('Line Chart')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Bar chart
categories = ['Q1', 'Q2', 'Q3', 'Q4']
values = [42, 58, 51, 67]
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
axes[0, 1].bar(categories, values, color=colors, edgecolor='white', linewidth=0.8)
axes[0, 1].set_title('Bar Chart')
axes[0, 1].set_ylabel('Revenue (k$)')
for i, v in enumerate(values):
    axes[0, 1].text(i, v + 0.5, str(v), ha='center', va='bottom', fontsize=10)

# Scatter chart
x_s = rng.normal(0, 1, 200)
y_s = 0.5 * x_s + rng.normal(0, 0.5, 200)
axes[0, 2].scatter(x_s, y_s, alpha=0.5, s=25, c=y_s, cmap='coolwarm')
axes[0, 2].set_title('Scatter Chart')
axes[0, 2].set_xlabel('x')
axes[0, 2].set_ylabel('y')

# Histogram
data_hist = rng.normal(50, 15, 500)
axes[1, 0].hist(data_hist, bins=30, color='mediumpurple', edgecolor='white', density=True)
axes[1, 0].set_title('Histogram (density)')
axes[1, 0].set_xlabel('Value')
axes[1, 0].set_ylabel('Density')

# Pie chart
sizes = [30, 25, 20, 15, 10]
labels = ['Product A', 'Product B', 'Product C', 'Product D', 'Other']
explode = (0.05, 0, 0, 0, 0)
axes[1, 1].pie(sizes, labels=labels, autopct='%1.1f%%', explode=explode,
               startangle=90, shadow=True)
axes[1, 1].set_title('Pie Chart')

# Area / fill_between
axes[1, 2].fill_between(x, np.sin(x), alpha=0.4, color='steelblue', label='sin area')
axes[1, 2].plot(x, np.sin(x), color='steelblue', linewidth=1.5)
axes[1, 2].axhline(0, color='black', linewidth=0.8)
axes[1, 2].set_title('Area Chart')
axes[1, 2].legend()

plt.suptitle('Matplotlib Chart Gallery', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Figure Customization

Fine-grained control over every visual element: titles, axis labels, tick formatting, legends, and styles.

In [ ]:
# Available styles
print('Available styles:', plt.style.available[:8], '...')

# Styled figure
with plt.style.context('seaborn-v0_8-darkgrid'):
    fig, ax = plt.subplots(figsize=(9, 4))

    months = np.arange(1, 13)
    sales_2023 = [120, 135, 150, 145, 160, 175, 190, 185, 170, 165, 180, 210]
    sales_2022 = [100, 110, 130, 125, 140, 155, 165, 160, 150, 145, 155, 190]

    ax.plot(months, sales_2023, 'o-', color='#2196F3', linewidth=2.5,
            markersize=7, label='2023', markerfacecolor='white', markeredgewidth=2)
    ax.plot(months, sales_2022, 's--', color='#FF9800', linewidth=2.0,
            markersize=7, label='2022', markerfacecolor='white', markeredgewidth=2)

    ax.fill_between(months, sales_2022, sales_2023, alpha=0.15, color='#2196F3')

    ax.set_title('Monthly Sales: 2022 vs 2023', fontsize=14, fontweight='bold', pad=12)
    ax.set_xlabel('Month', fontsize=11)
    ax.set_ylabel('Revenue (k$)', fontsize=11)
    ax.set_xticks(months)
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                         'Jul','Aug','Sep','Oct','Nov','Dec'])
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f'${v:.0f}k'))
    ax.legend(fontsize=11, framealpha=0.9)
    ax.set_ylim(80, 230)

    plt.tight_layout()
    plt.show()

## 3. Seaborn Statistical Plots

Seaborn sits on top of Matplotlib and adds **statistical semantics**: it understands DataFrames, automatically computes confidence intervals, and groups data by categorical variables.

In [ ]:
# Create a rich dataset for seaborn demos
np.random.seed(1)
n = 300
categories = rng.choice(['Group A', 'Group B', 'Group C'], n)
df = pd.DataFrame({
    'group':  categories,
    'score':  np.where(categories == 'Group A',
                  rng.normal(70, 10, n),
              np.where(categories == 'Group B',
                  rng.normal(80, 12, n),
                  rng.normal(75, 8, n))),
    'hours_studied': rng.uniform(1, 10, n),
    'gender': rng.choice(['Male', 'Female'], n)
})
df['score'] = df['score'].clip(0, 100).round(1)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# histplot with KDE
sns.histplot(data=df, x='score', hue='group', kde=True, bins=20,
             ax=axes[0,0], palette='Set2', alpha=0.6)
axes[0,0].set_title('Histogram + KDE by Group')

# boxplot
sns.boxplot(data=df, x='group', y='score', palette='Set2', ax=axes[0,1],
            linewidth=1.5)
axes[0,1].set_title('Boxplot')

# violinplot
sns.violinplot(data=df, x='group', y='score', hue='gender',
               split=True, palette='pastel', ax=axes[0,2], inner='quart')
axes[0,2].set_title('Split Violinplot')

# scatterplot with regression
sns.scatterplot(data=df, x='hours_studied', y='score', hue='group',
                palette='Set2', alpha=0.6, ax=axes[1,0])
axes[1,0].set_title('Scatter: Hours Studied vs Score')

# barplot with CI
sns.barplot(data=df, x='group', y='score', hue='gender',
            palette='Set1', ax=axes[1,1], capsize=0.1)
axes[1,1].set_title('Barplot with 95% CI')

# correlation heatmap
corr = df[['score', 'hours_studied']].assign(
    score_sq=lambda x: x['score']**2
).corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            ax=axes[1,2], vmin=-1, vmax=1, linewidths=0.5)
axes[1,2].set_title('Correlation Heatmap')

plt.suptitle('Seaborn Statistical Plot Gallery', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Pairplot and FacetGrid

`pairplot` creates a grid of pairwise scatter plots and histograms — great for quickly exploring relationships in multi-variable datasets. `FacetGrid` is the underlying engine for any faceted plot.

In [ ]:
# Pairplot
g = sns.pairplot(df[['score', 'hours_studied', 'group']],
                 hue='group', palette='Set2',
                 plot_kws={'alpha': 0.5, 's': 20},
                 diag_kind='kde')
g.figure.suptitle('Pairplot', y=1.02, fontsize=14)
plt.show()

# FacetGrid: histogram of score faceted by group
g2 = sns.FacetGrid(df, col='group', row='gender', palette='Set2',
                   height=3, aspect=1.2)
g2.map(sns.histplot, 'score', bins=15, color='steelblue', kde=True)
g2.set_axis_labels('Score', 'Count')
g2.set_titles(col_template='{col_name}', row_template='{row_name}')
g2.figure.suptitle('Score Distribution by Group and Gender', y=1.02, fontsize=13)
plt.show()

## 5. Color Palettes

In [ ]:
palettes = ['deep', 'muted', 'pastel', 'bright', 'dark', 'colorblind',
            'Set1', 'Set2', 'viridis', 'coolwarm']

fig, axes = plt.subplots(len(palettes), 1, figsize=(12, len(palettes) * 0.7))

for ax, name in zip(axes, palettes):
    try:
        colors = sns.color_palette(name, 8)
    except Exception:
        colors = sns.color_palette(name, as_cmap=False)
    for i, c in enumerate(colors[:8]):
        ax.add_patch(plt.Rectangle((i, 0), 1, 1, color=c))
    ax.set_xlim(0, 8)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.text(-0.2, 0.5, name, va='center', ha='right', fontsize=9)

plt.suptitle('Seaborn Color Palettes', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Saving Figures

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.linspace(0, 2 * np.pi, 500)
ax.plot(x, np.sin(x) * np.exp(-0.2 * x), linewidth=2.5, color='teal')
ax.fill_between(x, np.sin(x) * np.exp(-0.2 * x), alpha=0.3, color='teal')
ax.set_title('Damped Sine Wave', fontsize=14)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.grid(True, alpha=0.3)
plt.tight_layout()

# Save as PNG (raster) and SVG (vector)
plt.savefig('/tmp/damped_sine.png', dpi=150, bbox_inches='tight')
plt.savefig('/tmp/damped_sine.svg', bbox_inches='tight')
print('Saved to /tmp/damped_sine.png and /tmp/damped_sine.svg')
plt.show()

## Practice Exercises

**Exercise 1 — Multi-panel Dashboard**
Using any dataset you like (or generate one), create a 2x2 subplot figure with: a line chart of a cumulative sum, a histogram, a scatter plot with a color-coded third variable, and a horizontal bar chart. Add a shared super-title.

**Exercise 2 — Seaborn Theme Comparison**
Plot the same `sns.boxplot` five times side-by-side using five different Seaborn themes (`whitegrid`, `darkgrid`, `white`, `dark`, `ticks`). Which do you find most readable?

**Exercise 3 — Annotated Chart**
Create a line chart of monthly temperatures for two cities over a year. Annotate the peak and trough of each city's curve using `ax.annotate()` with arrows. Add a shaded region between the two lines using `fill_between`.